# Module 11 — RAG v1: Naive RAG

Companion notebook to [`docs/modules/11_rag_v1_naive_rag.md`](../docs/modules/11_rag_v1_naive_rag.md).

Every stage through prompt assembly runs for real: loading the 20-file Nimbus handbook corpus,
chunking (Module 8's `chunk_text()`), embedding (Module 9's `FakeEmbedder`), storing and
retrieving (Module 10's `NumpyVectorStore`), and citation-tagged prompt assembly. Answer
generation uses Module 6's `FakeRuntime` with a scripted response, since a live LLM runtime
isn't installed on this machine (see the theory doc's machine note) — `NaiveRagPipeline`
accepts any real `LLMRuntime` unchanged.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_11"))
CORPUS_DIR = REPO_ROOT / "datasets" / "rag_docs" / "nimbus_handbook"
print(f"Repo root: {REPO_ROOT}")
print(f"Corpus files: {len(list(CORPUS_DIR.glob('*.md')))}")


Repo root: /Users/bhakti/workspace/local_ai_app
Corpus files: 20


## 1. Ingest the corpus: load -> chunk -> embed -> store

In [2]:
from local_ai_core.runtimes.fake import FakeRuntime
from local_ai_rag.embeddings.fake import FakeEmbedder
from local_ai_rag.pipeline import NaiveRagPipeline
from local_ai_rag.stores.numpy_store import NumpyVectorStore

embedder = FakeEmbedder(dimensions=64)
store = NumpyVectorStore()
runtime = FakeRuntime(default_response="The password reset link expires in 15 minutes [password_reset::0].")
pipeline = NaiveRagPipeline(embedder, store, runtime, model="fake-model", chunk_max_chars=500)

chunks = await pipeline.ingest_directory(CORPUS_DIR)
print(f"{len({c.doc_id for c in chunks})} documents -> {len(chunks)} chunks")


20 documents -> 38 chunks


## 2. Retrieve for a real question

In [3]:
question = "How long does a password reset link stay valid?"
results = await pipeline.retrieve(question, k=3)
for r in results:
    print(f"{r.score:.3f}  {r.doc_id:25s}  {r.text[:70]}...")


0.383  password_reset::0          If you forget your Nimbus password, go to the sign-in page and select ...
0.377  account_creation::0        To create a Nimbus Cloud Storage account, visit signup.nimbus.example ...
0.345  subscription_plans::1      All paid plans include a 14-day free trial; no credit card is required...


## 3. Prompt assembly (curriculum's minimal RAG prompt, verbatim)

In [4]:
from local_ai_rag.context_packers.citation_packer import build_rag_prompt

prompt = build_rag_prompt(question, results)
print(prompt)


You are a question answering assistant.
Answer only using the provided context.
If the answer is not present in the context, say: "I don't know based on the provided documents."

Context:
[password_reset::0] If you forget your Nimbus password, go to the sign-in page and select "Forgot password."
Enter the email address associated with your account and Nimbus will send a password reset
link. The reset link expires in 15 minutes for security reasons; if it expires, request a new
one from the same page.

After resetting your password, all active sessions on all devices are signed out
automatically, and you will need to sign in again everywhere.

[account_creation::0] To create a Nimbus Cloud Storage account, visit signup.nimbus.example and provide a valid
email address, a display name, and a password of at least 10 characters. Nimbus sends a
verification email immediately; the verification link expires after 24 hours. Accounts
created without verifying email within 24 hours are automatica

## 4. Answer generation and citation grounding — Lab 2

Generation itself is `FakeRuntime` (honest-skip for a real model, see the machine note), but
citation extraction and grounding verification are real code operating on real strings.

In [5]:
answer = await pipeline.answer(question, k=3)
print("Answer:", answer.answer_text)
print("Citations:", answer.citations)
print("Citations grounded in retrieved chunks:", answer.citations_are_grounded)


Answer: The password reset link expires in 15 minutes [password_reset::0].
Citations: ['password_reset::0']
Citations grounded in retrieved chunks: True


## 5. Detecting an invented citation — the gotcha made measurable

A citation the model didn't actually retrieve is a real small-model failure mode. Here it's
provoked deliberately with a scripted `FakeRuntime` response to prove the detection code works,
not to claim a real model does this — that's the resourced Mac's job to observe.

In [6]:
bad_runtime = FakeRuntime(default_response="According to our records [totally_invented_doc::4].")
bad_pipeline = NaiveRagPipeline(embedder, store, bad_runtime, model="fake-model", chunk_max_chars=500)

bad_answer = await bad_pipeline.answer(question, k=3)
print("Citations:", bad_answer.citations)
print("Citations grounded:", bad_answer.citations_are_grounded)


Citations: ['totally_invented_doc::4']
Citations grounded: False


## 6. Answerable vs. unanswerable questions, and retrieval quality — Labs 3-4

In [7]:
from qa_eval import evaluate_answerable, evaluate_unanswerable

answerable_metrics = await evaluate_answerable(pipeline, k=3)
print("Answerable questions, doc-level retrieval quality:")
for key, value in answerable_metrics.items():
    print(f"  {key}: {value:.2f}")

print()
print("Unanswerable questions, top retrieval score (no golden match exists in the corpus):")
for row in await evaluate_unanswerable(pipeline, k=3):
    print(f"  {row['top_score']:.3f}  {row['question']}")


Answerable questions, doc-level retrieval quality:
  mean_recall_at_k: 0.62
  mean_precision_at_k: 0.25
  mrr: 0.62
  mean_ndcg_at_k: 0.69

Unanswerable questions, top retrieval score (no golden match exists in the corpus):
  0.387  Who is the CEO of Nimbus?
  0.487  Does Nimbus offer a student discount?
  0.436  What is Nimbus's carbon offset program?
  0.507  What is the maximum number of workspace members on the Enterprise plan?


## 7. Chunk size vs. retrieval quality — Lab 5

Real, measured trade-off: chunking too aggressively (150 chars) genuinely hurts retrieval
quality on this corpus, not an assumed result.

In [8]:
from compare_chunk_sizes import compare_chunk_sizes, results_to_markdown_table

chunk_size_results = await compare_chunk_sizes([150, 500, 1200], k=3)
print(results_to_markdown_table(chunk_size_results))


# Lab 5 — chunk size vs. retrieval quality

| Chunk size (chars) | Chunks in index | Recall@k | Precision@k | MRR | nDCG@k |
|---:|---:|---:|---:|---:|---:|
| 150 | 100 | 0.38 | 0.12 | 0.29 | 0.31 |
| 500 | 38 | 0.62 | 0.25 | 0.62 | 0.69 |
| 1200 | 20 | 0.62 | 0.21 | 0.62 | 0.62 |

